# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

# Task type: Clustering (unsupervised)

My lane (Structured Content Archetype Clustering) maps onto the ML loop as a clustering
task — not classification, ranking, or scoring. There is no existing label that says "this
page belongs to archetype X." Instead, the goal is to group pages that behave alike across
several observable metrics (impressions, CTR, position, engagement, freshness, word count)
and let the structure in the data reveal a small number of interpretable groups.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [8]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Task type: Clustering (unsupervised)")
print(f"Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns")

Task type: Clustering (unsupervised)
Dataset ready: 30000 rows, 44 columns


## 2. Target or proxy

Target or proxy: There is no true target column, since clustering is unsupervised. The
"proxy" here is cluster membership itself — an integer label assigned by the clustering
algorithm (e.g. K-Means) based on distance in feature space. Unlike a classification label,
this proxy has no ground truth to check against; it only becomes meaningful once I inspect
real pages inside each cluster and give it an interpretable name (e.g. "champions,"
"stale visible pages").

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sente# Sanity check: confirm no existing archetype/label column exists in the raw data
print("archetype" in df.columns.str.lower().tolist())  # should be False — proxy must be built, not borrowednces here breaks Run All.


False


## 3. Success metric

Success metric: Clustering has no accuracy/precision like supervised tasks. I'll use:
- Silhouette score — measures how well-separated and internally consistent clusters are.
- Cluster stability — do similar archetypes reappear with a different random seed?
- Human-readability — can each cluster's typical profile be described in one sentence a
  content reviewer would act on? This qualitative check matters most for this lane's decision.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import silhouette_score
print("Metric of record: silhouette_score (from sklearn.metrics)")
print("Secondary checks: cluster stability across seeds, human-readable cluster profiles")

Metric of record: silhouette_score (from sklearn.metrics)
Secondary checks: cluster stability across seeds, human-readable cluster profiles


## 4. The unit of analysis, as a real dataframe

One row = one content page, represented by a snapshot of its search and engagement metrics
over the last 90 days. This is the grain clustering will operate on — I am not clustering
clients, queries, or days, just individual pages.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = df[["content_id", "impressions_90d", "ctr", "avg_position",
                        "engagement_rate", "word_count", "content_age_days"]]
print(f"{len(unit_of_analysis):,} rows — each row is ONE content page")
unit_of_analysis.head()

30,000 rows — each row is ONE content page


,content_id,impressions_90d,ctr,avg_position,engagement_rate,word_count,content_age_days
0,content_304f48230142,3803,0.76,10.6,5.88,3221.0,187
1,content_a1fb4e703a9e,15320,0.05,20.3,0.00,2481.0,445
2,content_9aa793d4d895,12581,0.09,36.5,0.00,3515.0,141
3,content_331d6c4de07b,11751,0.49,6.2,1.28,NaN,463
4,content_d99b7a2d90ca,19140,0.13,44.0,0.00,2803.0,263


## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "if impressions > 1000 and word_count < 1000, call it thin-but-visible")
can only check conditions I think to write down in advance, one dimension at a time. But
Notebook 01 already showed single metrics don't tell the whole story — search volume barely
predicts traffic (corr = 0.001), and word count doesn't separate declining from growing pages.
Real archetypes likely emerge from combinations of several metrics moving together in ways
I wouldn't guess by hand. Clustering finds these combinations automatically from the data's
structure, rather than me hard-coding a guess about which combinations matter.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Evidence: single metrics are weak predictors on their own (from Notebook 01)
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"search_volume vs impressions_90d correlation: {corr:.3f} — too weak for a single-metric rule")

search_volume vs impressions_90d correlation: 0.001 — too weak for a single-metric rule


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.